# Base Place Recognition Pipeline 

Test Place Recognition on the 3DSSG dataset using `opr.pipelines`

In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/home/kartashov_ga/projects/GSLoc/.venv/lib/python3.10/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:1000: The handshake operation timed out>
  data = fetch_version_info()
2026-03-25 14:45:17.153 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


## Create dataset object

In [2]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
index_path = "/home/kartashov_ga/projects/tests/gsloc/26-03-25/"

In [3]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [ ]:
# dataloaders = {}
# 
# That was example of how to init ITLP to understand what can be set 
# 
# for track in TRACK_LIST:
#     dataset = ITLPCampus(
#         dataset_root=f"{dataset_path}/{track}",
#         sensors=["front_cam"],
#         mink_quantization_size=0.5,
#         max_point_distance=40.0,
#         image_transform=image_transform_fn,
#         load_semantics=False,
#         load_text_descriptions=False,
#         load_text_labels=False,
#         load_aruco_labels=False,
#         indoor=True,
#     )
#     dataloaders[track] = DataLoader(
#         dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=dataset.collate_fn
#     )


In [9]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index_path,
    rebuild_meta=False,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False
)
# You can create your own dataloader for index generation
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

## Create model

In [10]:
model = MegaLoc()
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

## Create Index (files that are used to do retrievel based on database)

In [ ]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    dataloader = None,
    model=model,
    rebuild_meta=False,
    rebuild_descriptors = False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-03-15 21:32:15.211 | INFO     | mmpr.inference.index:generate:391 - Using existing meta.parquet
2026-03-15 21:32:15.212 | INFO     | mmpr.inference.index:generate:418 - Using existing descriptors.npy
2026-03-15 21:32:15.331 | INFO     | mmpr.inference.index:generate:436 - schema.json file was saved in /home/kartashov_ga/projects/tests/gsloc/14-03-26/


Index created at /home/kartashov_ga/projects/tests/gsloc/14-03-26/
Index size: 20000, dim: 8448 metric: l2


# Test PlaceRecognitionPipeline

In [8]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
)

In [9]:
three_rscan_q = ThreeRScan(
    dataset_root="/mnt/external_usb_hdd/6YL/Datasets/3RScan",
    # meta_path = index_path,
    rebuild_meta=True,
    limit=10000,
    image_transform=image_transform_fn
)

2026-03-15 21:33:03.692 | INFO     | gsloc.datasets.three_rscan:__init__:147 - Rebuilding metadata for 3rscan dataset
2026-03-15 21:33:03.692 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:90 - Scanning 3rscan dataset...
2026-03-15 21:33:06.656 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:118 - Scanned 10000 rows


In [10]:
out = pipeline.infer(three_rscan_q[9000])

In [11]:
out

PlaceRecognitionResult(descriptor=array([ 0.00143673,  0.00783881,  0.02143787, ...,  0.01978837,
       -0.00375643, -0.0045516 ], shape=(8448,), dtype=float32), indices=array([9000, 8787, 8786, 9001, 9005]), distances=array([8.8346681e-09, 5.0083792e-01, 7.0701224e-01, 7.5864244e-01,
       8.1422341e-01], dtype=float32), db_idx=array([9000, 8787, 8786, 9001, 9005]), db_pose=array([[ 0.798643  ,  0.983432  , -0.132033  ,  0.75250036, -0.5780175 ,
        -0.25381622, -0.18766014],
       [ 0.356377  ,  1.48875   , -0.131014  ,  0.8410546 , -0.427336  ,
        -0.3099954 , -0.11795727],
       [ 0.382755  ,  1.40634   , -0.0807452 ,  0.8403163 , -0.41481936,
        -0.32412416, -0.12937145],
       [ 0.841705  ,  0.995071  , -0.140846  ,  0.7504945 , -0.58146584,
        -0.24821363, -0.19247201],
       [ 0.860654  ,  0.989988  , -0.125784  ,  0.7427527 , -0.56332934,
        -0.28782725, -0.21939473]], dtype=float32))

In [ ]:
from mmpr.pr_infer import PRInferConfig, PRInferencer
from mmpr.seq_pr_benchmark import SequenceBenchmarkConfig, SequencePRBenchmarker

FORCE_RERUN = False
PER_FRAME_K = 10   # Top-k per frame for sequence PR

pr_cache_path = index_path

if pr_cache_path.exists() and not FORCE_RERUN:
    print(f"Using existing PR cache: {pr_cache_path}")
else:
    cfg_inf = PRInferConfig(
        root_data_dir=root_data_dir,
        map_name=map_name,
        df=q_df,
        index_dir=index_path,
        device="cuda",
        per_frame_k=PER_FRAME_K,
        model=model
    )
    
    PRInferencer(cfg_inf).save(pr_cache_path)
    print(f"Built PR cache in {pr_cache_path}")


def build_pr_cache_for_map(map_name: str, q_df: pd.DataFrame) -> Path:
    """Build or load PR cache for a given map.

    Args:
        map_name: Map identifier such as "map2".
    Returns:
        Path to NPZ cache file.
    """
    


def _configs_match_json(d: dict, cfg: SequenceBenchmarkConfig) -> bool:
    """Return True if saved metrics.json config matches the benchmark config."""
    conf = d.get("config", {})
    try:
        return (
            int(conf.get("max_window", -1)) == int(cfg.max_window)
            and int(conf.get("per_frame_k_used", -1)) == int(cfg.per_frame_k_used)
            and int(conf.get("final_k", -1)) == int(cfg.final_k)
            and str(conf.get("recency_weighting", "")) == str(cfg.recency_weighting)
            and abs(float(conf.get("recall_threshold_m", -1.0)) - float(cfg.recall_threshold_m)) < 1e-9
        )
    except Exception:
        return False


def _row_from_metrics_json(d: dict, W: int, map_name: str) -> dict:
    """Convert metrics.json payload to a single summary row."""
    rk = d.get("recall_at_k", {}) or {}
    return {
        "w": int(W),
        "auc_pr": float(d.get("auc_pr", 0.0)),
        "f1_max": float(d.get("f1_max", 0.0)),
        "recall_at_1": float(rk.get("1", 0.0)),
        "recall_at_5": float(rk.get("5", 0.0)),
        "recall_at_10": float(rk.get("10", 0.0)),
        "recall_at_25": float(rk.get("25", 0.0)),
        "num_valid": int(d.get("num_queries_valid", 0)),
        "num_total": int(d.get("num_queries_total", 0)),
        "query_track": map_name,
    }


def run_sweep(
    q_df: pd.DataFrame,
    map_name: str,
    pr_cache_path: Path,
    seq_lengths: Iterable[int] = [MAX_WINDOW]
) -> pd.DataFrame:
    """Run or reuse sequence benchmark for a map across sequence lengths."""
    all_rows: list[dict] = []
    for W in tqdm(list(seq_lengths)):
        out_dir = OUTPUT_DIR / f"{map_name}_w{W:03d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        cfg_b = SequenceBenchmarkConfig(
            q_df=q_df,
            db_index_dir=index_dir,
            cache_path=pr_cache_path,
            root_data_dir=root_data_dir,
            map_name=map_name,
            max_window=int(W),
            per_frame_k_used=PER_FRAME_K,
            final_k=FINAL_K,
            recency_weighting="none",
            recall_threshold_m=0.5,
        )
        # metrics_path = out_dir / "metrics.json"

        # if SKIP_IF_EXISTS and metrics_path.exists() and not FORCE_RERUN:
        #     try:
        #         d = json.loads(metrics_path.read_text())
        #         if _configs_match_json(d, cfg_b):
        #             all_rows.append(_row_from_metrics_json(d, W, map_name))
        #             continue
        #     except Exception:
        #         pass

        bench = SequencePRBenchmarker(cfg_b)
        artifacts = bench.run()
        bench.save(artifacts, out_dir)
        all_rows.append({
            "w": int(W),
            "auc_pr": float(artifacts.auc_pr),
            "f1_max": float(artifacts.f1_max),
            "recall_at_1": float(artifacts.recall_at_k.get(1, 0.0)),
            "recall_at_5": float(artifacts.recall_at_k.get(5, 0.0)),
            "recall_at_10": float(artifacts.recall_at_k.get(10, 0.0)),
            "recall_at_25": float(artifacts.recall_at_k.get(25, 0.0)),
            "num_valid": int(artifacts.num_queries_valid),
            "num_total": int(artifacts.num_queries_total),
            "query_track": map_name,
        })

    df = pd.DataFrame(all_rows).sort_values("w").reset_index(drop=True)
    return df


In [ ]:
# Run benchmarks for configured maps; save per-map and combined summaries
summaries: dict[str, pd.DataFrame] = {}
m = "first-10k-on-first-10k"

print(f"=== {m} ===")
cache_path = build_pr_cache_for_map(m, q_df[:10000])
df_m = run_sweep(q_df[:10000], m, cache_path, seq_lengths=list(range(1,21)))
summaries[m] = df_m
# Save per-map summary
out_map_dir = OUTPUT_DIR / m
out_map_dir.mkdir(parents=True, exist_ok=True)
(out_map_dir / "summary.csv").write_text(df_m.to_csv(index=False))

# Combined summary across maps
summary_all = pd.concat(list(summaries.values()), ignore_index=True)
(OUTPUT_DIR / "summary_all.csv").write_text(summary_all.to_csv(index=False))

display(summary_all.head(3))
display(summary_all.tail(3))